<a href="https://colab.research.google.com/github/apmontesp/Landslides_-Applied-ML-Course/blob/main/notebooks/01_eda_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 01: Análisis Exploratorio de Datos (EDA) - Landslide4Sense
**Proyecto:** Detección de Deslizamientos mediante Inteligencia Artificial  
**Institución:** Universidad EAFIT  

Este notebook aborda la preparación del entorno, la ingesta de datos multimodales (Óptico, SAR, DEM) y el análisis estadístico del balance de clases.

## 1. Configuración del Entorno y Adquisición de Datos
Instalación de dependencias de visión artificial y descarga del dataset desde la API de Kaggle.

In [ ]:
from google.colab import drive
import subprocess, sys, os, h5py
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from tqdm import tqdm

# Instalación de librerías
packages = ['h5py', 'tqdm', 'matplotlib', 'seaborn']
for pkg in packages:
    subprocess.run([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

# Montar Drive
drive.mount('/content/drive')

# Rutas de datos (mismo patrón que notebooks 02-06)
base_path = Path('/content/drive/MyDrive/Landslide4Sense')
img_dirs  = list(base_path.glob('**/TrainData/img'))

if img_dirs:
    train_img_dir  = img_dirs[0]
    train_mask_dir = train_img_dir.parent / 'mask'
    DATA_ROOT  = train_img_dir.parent.parent
    img_list   = sorted(list(train_img_dir.glob('*.h5')))
    mask_list  = sorted(list(train_mask_dir.glob('*.h5')))
    print(f'Dataset : {DATA_ROOT}')
    print(f'Pares detectados: {len(img_list)}')
else:
    print('ERROR: No se encontro TrainData en Drive.')
    print('Verifica que exista: MyDrive/Landslide4Sense/TrainData/img/')


## 2. Análisis de Balance de Clases y Morfología
Evaluación del desbalance entre píxeles de deslizamiento y suelo estable.

In [ ]:
import numpy as np
import h5py
import matplotlib.pyplot as plt
from tqdm import tqdm

labels, areas = [], []

for mf in tqdm(mask_list, desc='Procesando máscaras'):
    with h5py.File(mf, 'r') as hf:
        mask = hf[list(hf.keys())[0]][()]
        labels.append(int(mask.max() > 0))
        areas.append(float(mask.mean()))

labels = np.array(labels)
areas = np.array(areas)

fig, ax = plt.subplots(1, 2, figsize=(15, 6))
n_pos, n_neg = labels.sum(), len(labels) - labels.sum()

ax[0].pie([n_pos, n_neg], labels=['Deslizamiento', 'Estable'], autopct='%1.1f%%', colors=['#E67E22', '#2E86C1'])
ax[0].set_title('Balance de Clases (Parches)')

ax[1].hist(areas[labels==1]*100, bins=30, color='#E67E22', alpha=0.7, edgecolor='black')
ax[1].set_title('Severidad: % de área afectada')
plt.show()

print(f'Relación Negativo/Positivo: {n_neg/n_pos:.2f}')

## 3. Dashboard Multimodal
Visualización de la respuesta espectral (RGB), Radar (SAR) y Topografía (DEM).

In [ ]:
def norm(data):
    p2, p98 = np.percentile(data, (2, 98))
    return np.clip((data - p2) / (p98 - p2), 0, 1)

indices = np.random.choice(len(img_list), 3, replace=False)
fig, axes = plt.subplots(3, 4, figsize=(20, 15))

for i, idx in enumerate(indices):
    with h5py.File(img_list[idx], 'r') as hf: patch = hf[list(hf.keys())[0]][()]
    with h5py.File(mask_list[idx], 'r') as hf: mask = hf[list(hf.keys())[0]][()]
    
    axes[i, 0].imshow(norm(patch[:,:,[3,2,1]]))
    axes[i, 0].set_title('Óptico (RGB)')
    
    axes[i, 1].imshow(norm(patch[:,:,0]), cmap='gray')
    axes[i, 1].set_title('Radar (SAR VV)')
    
    axes[i, 2].imshow(patch[:,:,13], cmap='terrain')
    axes[i, 2].set_title('Elevación (DEM)')
    
    axes[i, 3].imshow(norm(patch[:,:,[3,2,1]]))
    axes[i, 3].imshow(mask, cmap='Reds', alpha=0.4)
    axes[i, 3].set_title('Ground Truth (Overlay)')
    
    for ax in axes[i]: ax.axis('off')

plt.tight_layout()
plt.show()